# TinyCU on a real GPU (Colab)

**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. Nothing needs editing.

What this does:
1. builds TinyCU and its C++ CUDA Driver API runner, runs the C++ tests;
2. runs the kernel suite: every kernel through NumPy, the interpreter, TinyCU's PTX on the GPU and
   the nvcc twin via NVRTC, all compared **bit-exactly**;
3. prints PTX/SASS statistics, the benchmark, and saxpy's PTX next to nvcc's;
4. zips `results/` and downloads it (commit those files to the repo afterwards).

In [ ]:
# 1. GPU check
!nvidia-smi --query-gpu=name,compute_cap,driver_version,memory.total --format=csv
!nvcc --version | tail -2

In [ ]:
# 2. Get the code (re-running this cell is safe)
import os
REPO = "https://github.com/kdt523/tinycu.git"
if not os.path.isdir("/content/tinycu"):
    !git clone -q {REPO} /content/tinycu
%cd /content/tinycu
!git log --oneline -1

In [ ]:
# 3. Build tools + build (Release; the GPU runner is built automatically when CUDA is found)
!apt-get -qq update > /dev/null && apt-get -qq install -y ninja-build > /dev/null
!cmake -S . -B build -G Ninja -DCMAKE_BUILD_TYPE=Release > /dev/null && cmake --build build 2>&1 | tail -1
!ls -la build/tinycu build/tinycu-run
!ctest --test-dir build 2>&1 | tail -3

In [ ]:
# 4. Python harness (CuPy is preinstalled on Colab; the [gpu] extra pins cupy-cuda12x)
!pip install -q -e "harness[gpu]"
import os
os.environ["TINYCU"] = "/content/tinycu/build/tinycu"
os.environ["PATH"] = "/usr/local/cuda/bin:" + os.environ["PATH"]

In [ ]:
# 5. GPU tests: GPU == NVRTC twin == interpreter == NumPy for every kernel and size
!python -m pytest -m gpu -q 2>&1 | tail -5

In [ ]:
# 6. Full kernel-suite report (written to results/)
!python -m tcharness.suite --gpu --out results/kernel_suite_colab.md > /tmp/suite.log; echo "exit code $?"
!grep -E "^- " results/kernel_suite_colab.md
!grep -E "MISMATCH|ERROR" results/kernel_suite_colab.md || echo "no mismatches"

In [ ]:
# 7. PTX / SASS statistics and the benchmark (TinyCU vs NVRTC -O3)
!python -m tcharness.ptxstats --out results/ptx_stats_colab.md | tail -4
!python -m tcharness.bench --out results/bench_colab.md | tail -12
!python -m tcharness.metrics 2>/dev/null || echo "(tcharness.metrics arrives in phase 7)"

In [ ]:
# 8. saxpy: TinyCU PTX vs nvcc PTX, side by side
import subprocess, html
from IPython.display import HTML, display
ours = subprocess.run(["build/tinycu", "tests/kernels/saxpy.tc"], capture_output=True, text=True).stdout
subprocess.run(["nvcc", "-ptx", "-arch=sm_75", "-fmad=false", "tests/kernels/saxpy.cu", "-o", "/tmp/saxpy_nvcc.ptx"], check=True)
theirs = open("/tmp/saxpy_nvcc.ptx").read()
col = lambda title, text: f"<div style='flex:1;min-width:0'><b>{title}</b><pre style='font-size:11px'>{html.escape(text)}</pre></div>"
display(HTML(f"<div style='display:flex;gap:16px'>{col('TinyCU', ours)}{col('nvcc -fmad=false', theirs)}</div>"))

In [ ]:
# 9. Download results/ (and copy to Google Drive if it is mounted)
import shutil, os
archive = shutil.make_archive("/content/tinycu_results", "zip", "/content/tinycu/results")
if os.path.isdir("/content/drive/MyDrive"):
    shutil.copy(archive, "/content/drive/MyDrive/")
    print("copied to Google Drive")
try:
    from google.colab import files
    files.download(archive)
except Exception as e:  # not running in Colab
    print("archive at", archive, "-", e)